# Cross-platform benchmark — CPU / GPU / FPGA

Compares the SAR-DDC compress/decompress pipeline across **CPU (x86)**, **GPU (CUDA, +CPU entropy)**
and **FPGA (ZCU102 DPU, +ARM entropy)** on one aligned schema.

- **Data**: `results/benchmark_unified/<model>/baseline_<scenario>_<platform>.json` (host, `benchmark_gpu.py`)
  + `results/benchmark_hardware/<model>/<config>_<scenario>.json` (FPGA, `benchmark_hardware`).
  Regenerate with `python scripts/benchmark/run_unified_benchmark.py --model-dir results/fpga/active_model/ --power`.
- **Quality**: FP32 PSNR/bpp from W&B (GPU/CPU, linked by `wandb_run_id`); INT8 PSNR/bpp from `metrics.json` (FPGA).
- Every plot is **argument-driven** — change the call (`series=[...]`, `scenario=`, `size_by_bpp=`, …) to regenerate.

**Fairness caveats** (carried in the data): host cycles the same 20-patch real subset as the FPGA;
CPU = x86 **6 threads** FP32; GPU energy = board + CPU-package (entropy runs on CPU); FPGA energy = MPSoC
SoC scope — compare **energy/inference**, not raw watts. GPU/CPU are FP32, FPGA is INT8 (so PSNR differs).

## Platform setup

| Platform | Hardware | Notes |
| --- | --- | --- |
| **CPU** | Intel Xeon w3-2423, x86 | `torch_threads=6`; FP32 inference |
| **GPU** | NVIDIA RTX A4000 | NN on CUDA (FP32); entropy coding on host CPU |
| **FPGA** | Xilinx ZCU102 DPU | NN on DPU (INT8); entropy coding + normalise on ARM CPU |

Energy scope: CPU = CPU package (RAPL); GPU = board power (GPU + CPU package); FPGA = full MPSoC SoC.
Compare **energy/inference**, not raw watts.

## §0 Setup

In [ ]:
import glob
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
sys.path.insert(0, str(ROOT / "notebooks"))
from _benchmark_loader import load_quality_metrics, load_runs, load_stage_breakdowns

FIG_DPI = 120
PLOTS_DIR = ROOT / "results" / "plots" / "cross_platform"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def savefig(name):
    plt.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")


# ---- palette ----
from _plotkit import PALETTE, ARCH_LABEL

PLAT = PALETTE["platforms"]  # fpga/gpu/cpu dynamic+idle
STAGE_CLR = PALETTE["cpp_stages"]
ARCH_ORDER = ["FP", "SHyp", "ResFP", "ResSHyp"]

# Stage stacking order + display labels (host_* shown without the prefix in legends)
STAGE_ORDER = [
    "normalize",
    "g_a",
    "h_a",
    "eb_compress",
    "eb_decompress",
    "h_s",
    "gc_compress",
    "gc_decompress",
    "g_s",
    "denorm",
    "host_concat_abs",
    "host_split_y_hat",
]
STAGE_DISPLAY = {"host_concat_abs": "concat_abs", "host_split_y_hat": "split_y_hat"}

# ---- SERIES registry: friendly name -> selection + style. Subset/reorder via `series=[...]`. ----
SERIES = {
    "CPU": dict(
        platform="cpu", config="baseline", color=PLAT["cpu_dynamic"], hatch="", label="CPU"
    ),
    "GPU": dict(
        platform="gpu", config="baseline", color=PLAT["gpu_dynamic"], hatch="", label="GPU"
    ),
    "FPGA-s0": dict(
        platform="fpga", config="s0", color=PLAT["fpga_dynamic"], hatch="", label="FPGA"
    ),
    "FPGA-s1": dict(
        platform="fpga", config="s1", color=PLAT["fpga_dynamic"], hatch="////", label="FPGA s1"
    ),
}
DEFAULT_SERIES = ["CPU", "GPU", "FPGA-s0", "FPGA-s1"]

# ---- load benchmark runs + stages (both trees) ----
df = load_runs(ROOT / "results/benchmark_hardware", ROOT / "results/benchmark_unified")
sb = load_stage_breakdowns(ROOT / "results/benchmark_hardware", ROOT / "results/benchmark_unified")
quality = load_quality_metrics(ROOT / "results/fpga/compiled_models")

# Energy-delay product (mJ·ms) — combined efficiency figure-of-merit (lower = fast AND low-energy).
df["edp_mJ_ms"] = df["energy_mJ_per_patch"] * df["total_latency_mean_ms"]

# ---- quality lookup: FP32 (W&B, by wandb_run_id) for gpu/cpu; INT8 (metrics.json) for fpga ----
# Both keyed by the SAME canonical metric names so any metric can be picked by its key
# (psnr_MERLIN, ssim_MERLIN, epd_MERLIN, mse_MERLIN, psnr_ADAM, ssim_ADAM, epd_ADAM, bpp, ...).
# Call quality_keys() to see what each precision actually has.
_NON_METRIC = {"model_name", "arch", "lambda", "seed"}
_INT8 = {
    mn: {k: v for k, v in row.items() if k not in _NON_METRIC}
    for mn, row in quality.set_index("model_name").iterrows()
}  # metrics.json (FPGA, INT8)
# W&B test_sub500/* -> canonical key (one-time source normalisation: the two eval sources differ)
_WANDB_MAP = {
    "psnr_MERLIN": "test_sub500/psnr_merlin",
    "ssim_MERLIN": "test_sub500/ssim_merlin",
    "ms_ssim_MERLIN": "test_sub500/ms_ssim_merlin",
    "epd_MERLIN": "test_sub500/epd_merlin",
    "mse_MERLIN": "test_sub500/mse_merlin",
    "psnr_ADAM": "test_sub500/psnr_adam_noc",
    "ssim_ADAM": "test_sub500/ssim_adam_noc",
    "epd_ADAM": "test_sub500/epd_adam_noc",
    "mse_ADAM": "test_sub500/mse_adam_noc",
    "bpp": "test_sub500/bpp",
    "enl_recon": "test_sub500/enl_recon",
}
_wandb = pd.read_csv(ROOT / "notebooks/SAR_DDC_FPGA_all_runs_WandB.csv").set_index("id")
_FP32 = {}
for f in glob.glob(str(ROOT / "results/fpga/compiled_models/*/manifest.json")):
    mn = Path(f).parent.name
    rid = json.load(open(f)).get("wandb_run_id")
    if rid in _wandb.index:
        r = _wandb.loc[rid]
        _FP32[mn] = {
            k: float(r[col]) for k, col in _WANDB_MAP.items() if col in r and pd.notna(r[col])
        }


def quality_of(model_name, platform, qmetric="psnr_MERLIN"):
    """Quality value by canonical key; FP32 (W&B) for gpu/cpu, INT8 (metrics.json) for fpga.

    Returns None if the metric is absent for that platform's eval source.
    """
    table = _INT8 if platform == "fpga" else _FP32
    v = table.get(model_name, {}).get(qmetric)
    return float(v) if v is not None and not pd.isna(v) else None


def quality_keys():
    """Canonical quality-metric keys available per precision (any can be passed as qmetric=)."""
    int8 = sorted(set().union(*[set(d) for d in _INT8.values()])) if _INT8 else []
    fp32 = sorted(set().union(*[set(d) for d in _FP32.values()])) if _FP32 else []
    return {"int8 (FPGA)": int8, "fp32 (GPU/CPU)": fp32}


def _row(arch, spec, scenario):
    """The single run row for an arch + a SERIES spec + scenario (or None)."""
    r = df[
        (df.arch == arch)
        & (df.platform == spec["platform"])
        & (df.config == spec["config"])
        & (df.scenario == scenario)
        & (df.model_name.str.contains("_L1000"))
    ]
    return r.iloc[0] if not r.empty else None


print(
    f"loaded {len(df)} runs ({sorted(df.platform.unique())}), {len(sb)} stage rows, "
    f"{len(quality)} quality rows, {len(_FP32)} FP32 W&B matches"
)
print("quality keys:", quality_keys())

In [ ]:
# Short series labels for paper-ready figures (hide the s0/s1 config jargon: both map to "FPGA").
PAPER_SERIES_LABEL = {"CPU": "CPU", "GPU": "GPU", "FPGA-s0": "FPGA", "FPGA-s1": "FPGA"}


## §1 Inventory & coverage

In [ ]:
def coverage(scenario="compress", series=DEFAULT_SERIES):
    """Completeness matrix (arch × series) for a scenario, + caveat reminders."""
    archs = [a for a in ARCH_ORDER if a in df.arch.values]
    grid = pd.DataFrame(index=archs, columns=series)
    for a in archs:
        for s in series:
            grid.loc[a, s] = "OK" if _row(a, SERIES[s], scenario) is not None else "—"
    print(f"Coverage — scenario={scenario}")
    print(grid.to_string())
    print(
        "\nCaveats: host=20-patch real subset, FP32, CPU 6 threads | FPGA INT8, MPSoC power scope |"
        " GPU power = board+CPU-package | compare energy/inference, not raw W."
    )


coverage("compress")

## §2–4 Latency / Throughput / Energy

`grouped_bars(metric, ...)` — one bar per `series` entry, grouped by arch. The improvement factor vs
`ref` (default CPU) is annotated above each non-ref bar. Edit `series=[...]`, `scenario=`, `archs=`.

In [ ]:
def grouped_bars(
    metric,
    ylabel,
    title,
    series=DEFAULT_SERIES,
    scenario="compress",
    archs=None,
    ref="CPU",
    lower_is_better=True,
    annotate_factor=True,
    paper_ready=False,
    subtitle=None,
    save=None,
):
    """Grouped bar chart across `series`, grouped by arch. metric is a column of `df`.

    paper_ready : drop the in-plot title (move to caption) and use short series labels (no s0/s1 jargon).
    subtitle     : informational note shown below the figure when paper_ready=False (e.g. energy scope).
    """
    archs = archs or [a for a in ARCH_ORDER if a in df.arch.values]
    x = np.arange(len(archs))
    n = len(series)
    w = 0.8 / n
    vals = {
        (a, s): (
            _row(a, SERIES[s], scenario)[metric]
            if _row(a, SERIES[s], scenario) is not None
            and not pd.isna(_row(a, SERIES[s], scenario)[metric])
            else None
        )
        for a in archs
        for s in series
    }
    ymax = max((v for v in vals.values() if v is not None), default=1)
    fig, ax = plt.subplots(figsize=(2.4 * len(archs) + 2, 5.8), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, lw=0.4, alpha=0.35, color="#aaa")
    for ai, a in enumerate(archs):
        refv = vals.get((a, ref))
        for si, s in enumerate(series):
            v = vals.get((a, s))
            if v is None:
                continue
            xp = x[ai] + (si - (n - 1) / 2) * w
            spec = SERIES[s]
            ax.bar(
                xp,
                v,
                width=w,
                color=spec["color"],
                hatch=spec["hatch"],
                edgecolor="white",
                lw=0.6,
                zorder=3,
            )
            ax.text(
                xp,
                v + ymax * 0.012,
                f"{v:.0f}" if v >= 10 else f"{v:.1f}",
                ha="center",
                va="bottom",
                fontsize=10,
                color="#222",
            )
            if annotate_factor and s != ref and refv:
                fac = (refv / v) if lower_is_better else (v / refv)
                ax.text(
                    xp,
                    v + ymax * 0.065,
                    f"{fac:.0f}×" if fac >= 10 else f"{fac:.1f}×",
                    ha="center",
                    va="bottom",
                    fontsize=9,
                    fontweight="bold",
                    color="#222",
                )
    ax.set_xticks(x)
    ax.set_xticklabels([ARCH_LABEL[a] for a in archs], fontsize=11)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, ymax * 1.18)
    if not paper_ready:
        ax.set_title(title, fontsize=12)
    ax.legend(
        handles=[
            Patch(
                facecolor=SERIES[s]["color"],
                hatch="" if paper_ready else SERIES[s]["hatch"],
                edgecolor="white",
                label=PAPER_SERIES_LABEL.get(s, s) if paper_ready else SERIES[s]["label"],
            )
            for s in series
        ],
        fontsize=10,
        framealpha=0.9,
    )
    if not paper_ready and subtitle:
        fig.text(0.5, -0.02, subtitle, ha="center", fontsize=8.5, style="italic", color="dimgray")
    fig.tight_layout()
    if save:
        savefig(save)
    plt.show()


grouped_bars(
    "total_latency_mean_ms",
    "latency / patch (ms)",
    "Total latency — compress (× = speedup vs CPU)",
    save="latency_compress",
)

In [ ]:
grouped_bars(
    "throughput_fps",
    "throughput (patches/s)",
    "Throughput — compress (× vs CPU)",
    lower_is_better=False,
    save="throughput_compress",
)

In [ ]:
# exploratory energy bars (all series)
grouped_bars(
    "energy_mJ_per_patch",
    "energy / patch (mJ)",
    "Energy per inference — compress (× = less energy vs CPU)",
    subtitle="GPU energy: board + CPU-package  |  FPGA energy: MPSoC SoC  |  compare energy/inference, not raw W",
    save="energy_compress",
)

# ---- F6 paper energy figure: total only (dynamic is too small to read as a stack), s1 as "FPGA" ----
grouped_bars(
    "energy_mJ_per_patch",
    "energy / patch (mJ)",
    "",
    series=["CPU", "GPU", "FPGA-s1"],
    paper_ready=True,
    save="fig_energy",
)


def energy_table(series=("CPU", "GPU", "FPGA-s1"), scenario="compress", archs=None, ref="CPU"):
    """LaTeX energy table — total (dynamic) mJ/patch per platform, × vs `ref`.

    Alternative to the energy figure (Fig. fig_energy): the dynamic share is hard to see as a stacked
    bar, so it is reported numerically here. Comment out either the figure or this table in the paper.
    """
    archs = archs or [a for a in ARCH_ORDER if a in df.arch.values]
    head = [PAPER_SERIES_LABEL.get(s, s) for s in series]
    lines = [
        r"\begin{tabular}{l" + "r" * len(series) + "}",
        r"\toprule",
        r"Arch & \multicolumn{%d}{c}{Energy / patch [mJ]: total (dynamic)} \\" % len(series),
        "     & " + " & ".join(head) + r" \\",
        r"\midrule",
    ]
    for a in archs:
        rref = _row(a, SERIES[ref], scenario)
        refv = rref["energy_mJ_per_patch"] if rref is not None else None
        cells = []
        for s in series:
            r = _row(a, SERIES[s], scenario)
            if r is None:
                cells.append("--")
                continue
            tot, dyn = r["energy_mJ_per_patch"], r["dynamic_energy_mJ_per_patch"]
            x = "" if (s == ref or not refv) else f" {refv / tot:.0f}$\\times$"
            cells.append(f"{tot:.0f} ({dyn:.0f}){x}")
        lines.append(f"{ARCH_LABEL[a]} & " + " & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    print("\n".join(lines))


energy_table()

## §5 Per-stage breakdown

`stage_breakdown(series=...)` — one stacked bar per series, per arch, shared Y. Shows the bottleneck
shift (GPU NN collapses → entropy dominates; CPU/FPGA NN-dominated for residual archs).

In [ ]:
def stage_breakdown(
    series=DEFAULT_SERIES,
    scenario="compress",
    archs=None,
    pct_labels=True,
    pct_min=0.08,
    paper_ready=False,
    save=None,
):
    """Stacked per-stage latency, one bar per series, per arch (shared Y).

    pct_labels  : annotate each stage's share of its bar when the segment >= `pct_min` of the total.
    paper_ready : drop the suptitle and use short series labels (no s0/s1 jargon) for the manuscript.
    archs       : subset/reorder, e.g. ["FP", "ResSHyp"] (default: all four present).
    """
    archs = archs or [a for a in ARCH_ORDER if a in df.arch.values]
    fig, axes = plt.subplots(
        1, len(archs), figsize=(4.2 * len(archs), 5), dpi=FIG_DPI, sharey=True
    )
    axes = np.atleast_1d(axes)
    present = []
    xlabels = [PAPER_SERIES_LABEL.get(s, s) if paper_ready else s for s in series]
    for ax, a in zip(axes, archs):
        for xi, s in enumerate(series):
            r = _row(a, SERIES[s], scenario)
            if r is None:
                continue
            st = sb[
                (sb.platform == r.platform)
                & (sb.model_name == r.model_name)
                & (sb.config == r.config)
                & (sb.scenario == scenario)
            ]
            total = st.mean_ms.sum()
            bottom = 0.0
            for stg in STAGE_ORDER:
                row = st[st.stage == stg]
                if row.empty:
                    continue
                h = row.mean_ms.iloc[0]
                ax.bar(
                    xi,
                    h,
                    bottom=bottom,
                    width=0.75,
                    color=STAGE_CLR.get(stg, "#999"),
                    edgecolor="white",
                    lw=0.3,
                    zorder=3,
                )
                if pct_labels and total and h / total >= pct_min:
                    ax.text(
                        xi,
                        bottom + h / 2,
                        f"{100 * h / total:.0f}%",
                        ha="center",
                        va="center",
                        fontsize=6.5,
                        color="white",
                        fontweight="bold",
                    )
                if stg not in present:
                    present.append(stg)
                bottom += h
            if bottom:
                ax.text(xi, bottom, f"{bottom:.0f}", ha="center", va="bottom", fontsize=7.5)
        ax.set_xticks(range(len(series)))
        ax.set_xticklabels(
            xlabels,
            fontsize=8.5,
            rotation=0 if paper_ready else 30,
            ha="center" if paper_ready else "right",
        )
        ax.set_title(ARCH_LABEL[a], fontsize=11)
        ax.set_axisbelow(True)
        # Draw the bottom axis line above the bars (zorder=3) so the lowest segment
        # doesn't hide it.
        ax.spines["bottom"].set_zorder(4)
        ax.yaxis.grid(True, lw=0.4, alpha=0.3, color="#aaa")
    axes[0].set_ylabel("latency / patch (ms)")
    handles = [
        Patch(facecolor=STAGE_CLR.get(s, "#999"), label=STAGE_DISPLAY.get(s, s)) for s in present
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=min(len(handles), 8),
        fontsize=7.5,
        bbox_to_anchor=(0.5, 0.0),
    )
    if not paper_ready:
        fig.suptitle(f"Per-stage latency by platform — {scenario} (shared Y)", fontsize=13)
    fig.tight_layout(rect=(0, 0.05, 1, 1.0 if paper_ready else 0.97))
    if save:
        savefig(save)
    plt.show()


# ---- F6 paper figure: CPU/GPU/FPGA (s1 shown as plain "FPGA"), % labels, all four archs ----
stage_breakdown(
    archs=["FP", "ResSHyp"],
    series=["CPU", "GPU", "FPGA-s1"],
    paper_ready=True,
    save="fig_latency_breakdown",
)

# exploratory default (keeps the s0 vs s1 split + suptitle)
stage_breakdown()


## §6 Quality vs cost

`quality_scatter(xmetric, qmetric=...)` — quality (y) vs a cost metric (x), one marker per (arch, series).

- **`qmetric`** picks the quality metric by its canonical key (default `psnr_MERLIN`). Values are
  precision-correct: **FP32** (W&B) for GPU/CPU, **INT8** (`metrics.json`) for FPGA. Call
  `quality_keys()` for the valid keys per precision (e.g. `psnr_MERLIN`, `ssim_MERLIN`, `epd_MERLIN`,
  `mse_MERLIN`, `psnr_ADAM`, `ssim_ADAM`, `epd_ADAM`, `bpp`).
- **`xmetric`** is any `df` cost column (`energy_mJ_per_patch`, `total_latency_mean_ms`, `edp_mJ_ms`).
- `size_by_bpp=True` scales circles by bitrate; pick a subset (`series=["CPU","GPU","FPGA-s1"]`) to
  declutter the overlapping FPGA-s0/s1 points.

In [ ]:
def quality_scatter(
    xmetric="energy_mJ_per_patch",
    xlabel="energy / patch (mJ)",
    qmetric="psnr_MERLIN",
    qlabel=None,
    series=DEFAULT_SERIES,
    scenario="compress",
    archs=None,
    size_by_bpp=True,
    annotate=True,
    save=None,
):
    """Quality (y, `qmetric`) vs a cost metric (x). One marker per (arch, series).

    `qmetric` is any canonical quality key (see quality_keys()); value is precision-correct
    (FP32 W&B for gpu/cpu, INT8 metrics.json for fpga). Circle size ∝ bpp if `size_by_bpp`.
    """
    archs = archs or ARCH_ORDER
    qlabel = qlabel or qmetric
    pts, bpps = [], []
    for a in archs:
        for s in series:
            spec = SERIES[s]
            r = _row(a, spec, scenario)
            if r is None or pd.isna(r[xmetric]):
                continue
            yval = quality_of(r.model_name, spec["platform"], qmetric)
            if yval is None:
                continue
            bpp = quality_of(r.model_name, spec["platform"], "bpp")
            pts.append((r[xmetric], yval, bpp, spec, f"{ARCH_LABEL[a]}/{s}"))
            if bpp:
                bpps.append(bpp)
    if not pts:
        print(
            f"No points: qmetric={qmetric!r} unavailable for the selected series. "
            f"Try one of {quality_keys()}"
        )
        return
    bmin, bmax = (min(bpps), max(bpps)) if bpps else (1.0, 2.0)

    def msize(bpp):
        if not size_by_bpp or not bpp or bmax == bmin:
            return 140
        return 60 + (bpp - bmin) / (bmax - bmin) * 320

    fig, ax = plt.subplots(figsize=(8.5, 6), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.grid(True, lw=0.4, alpha=0.35, color="#aaa")
    for xv, yval, bpp, spec, lab in pts:
        ax.scatter(
            xv,
            yval,
            s=msize(bpp),
            color=spec["color"],
            hatch=spec["hatch"],
            edgecolor="white",
            lw=0.9,
            zorder=3,
            alpha=0.9,
        )
        if annotate:
            ax.annotate(lab, (xv, yval), textcoords="offset points", xytext=(7, 3), fontsize=7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"{qlabel}  [GPU/CPU: FP32 · FPGA: INT8]")
    title = f"{qlabel} vs {xlabel.split('(')[0].strip()} — {scenario}"
    if size_by_bpp:
        title += "  (circle ∝ bpp)"
    ax.set_title(title, fontsize=12)
    plat_h = [
        Patch(
            facecolor=SERIES[s]["color"],
            hatch=SERIES[s]["hatch"],
            edgecolor="white",
            label=SERIES[s]["label"],
        )
        for s in series
    ]
    leg1 = ax.legend(handles=plat_h, fontsize=7.5, loc="lower right", framealpha=0.9)
    ax.add_artist(leg1)
    if size_by_bpp and bmax > bmin:
        size_h = [
            Line2D(
                [0],
                [0],
                marker="o",
                color="#888",
                linestyle="None",
                markersize=(msize(b) ** 0.5) / 2.5,
                label=f"bpp≈{b:.2f}",
            )
            for b in (bmin, (bmin + bmax) / 2, bmax)
        ]
        ax.legend(
            handles=size_h,
            fontsize=7.5,
            loc="lower right",
            bbox_to_anchor=(0.72, 0.0),
            title="bitrate",
            framealpha=0.9,
        )
    fig.tight_layout()
    if save:
        savefig(save)
    plt.show()


# default: PSNR vs energy / latency
quality_scatter("energy_mJ_per_patch", "energy / patch (mJ)", save="psnr_vs_energy_compress")
quality_scatter("total_latency_mean_ms", "latency / patch (ms)", save="psnr_vs_latency_compress")
# switch the quality metric via qmetric= (see quality_keys() for valid keys):
quality_scatter("energy_mJ_per_patch", "energy / patch (mJ)", qmetric="ssim_MERLIN", qlabel="SSIM")
# declutter: quality_scatter(..., series=["CPU", "GPU", "FPGA-s1"], size_by_bpp=False)

## Notes / next

- Swap `scenario="full"` in any call for the full encode+decode pipeline.
- `series=["CPU","GPU","FPGA-s1"]` etc. to focus; `size_by_bpp=False` to drop bitrate sizing.
- `qmetric=` on `quality_scatter` switches the quality axis (see `quality_keys()`).
- TODO (future): seed-averaged quality (mean±std over the 6 seeds); throughput-per-watt; `full`-scenario figures.

## §7 Energy-delay product (EDP)

**Why:** latency and energy each tell only half the story — the GPU wins latency but draws more power;
the FPGA sips power but is slower. **Energy-delay product** `EDP = energy/patch × latency/patch`
(mJ·ms) is the standard hardware-efficiency figure-of-merit that rewards being *both* fast *and*
low-energy (a platform can't win EDP by trading one for the other). Lower is better.

**Insight it surfaces:** which platform is the best *overall* compute substrate for this workload once
you refuse to trade latency against energy — and how `s0→s1` channel-parallelism moves the FPGA on
that combined axis. (Reuses `grouped_bars` on the derived `edp_mJ_ms` column — same argument-driven API.)

In [ ]:
grouped_bars(
    "edp_mJ_ms",
    "energy-delay product (mJ·ms)",
    "Energy-delay product — compress (× = better vs CPU; lower EDP = fast AND low-energy)",
    save="edp_compress",
)